# RazorMind AI: Payment Success Prediction & Revenue Intelligence
### Inspired by Razorpay Optimizer & Merchant Analytics
**Objective**:
1. Predict transaction failure probabilities due to issuer bank downtime and channel latency.
2. Simulate Smart Routing to dynamically redirect transactions and uplift merchant success rates.
3. Perform RFM (Recency, Frequency, Monetary) customer churn segmentation and revenue drop attribution.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report
import joblib

data_path = os.path.join("..", "data", "raw", "transactions.csv")
df = pd.read_csv(data_path)
print(f"Transactions: {len(df)}")
df.head()

## 2. Bank Downtimes & Payment Gateway Failure Analysis

In [ ]:
# Analyze failure rate by bank
bank_analysis = df.groupby('bank_code').agg(
    total_tx=('transaction_id', 'count'),
    failed_tx=('status', lambda x: (x == 'failed').sum()),
    lost_revenue=('amount', lambda x: x[df.loc[x.index, 'status'] == 'failed'].sum())
).reset_index()

bank_analysis['failure_rate'] = (bank_analysis['failed_tx'] / bank_analysis['total_tx']) * 100
bank_analysis['success_rate'] = 100 - bank_analysis['failure_rate']
bank_analysis = bank_analysis.sort_values('failure_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=bank_analysis, x='bank_code', y='failure_rate', palette='Reds_r', ax=axes[0])
axes[0].set_title("Payment Failure Rate by Bank (%)", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Failure Rate (%)")

sns.barplot(data=bank_analysis, x='bank_code', y='lost_revenue', palette='Oranges_r', ax=axes[1])
axes[1].set_title("Lost Revenue due to Payment Failures (INR)", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Lost GMV (INR)")

plt.tight_layout()
plt.show()
bank_analysis

## 3. Training the Payment Success / Failure Model

In [ ]:
FEATURES = ["amount", "hour_of_day", "day_of_week", "bank_code", "payment_method"]
NUMERIC_FEATURES = ["amount", "hour_of_day", "day_of_week"]
CATEGORICAL_FEATURES = ["bank_code", "payment_method"]

df_target = df.copy()
df_target["is_failed"] = (df_target["status"] == "failed").astype(int)

X = df_target[FEATURES]
y = df_target["is_failed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
    ]
)

failure_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42))
])

failure_pipeline.fit(X_train, y_train)
y_prob = failure_pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)

print(f"Payment Failure Prediction ROC-AUC: {auc:.4f}")

## 4. Smart Routing Simulation (Razorpay Optimizer)
Simulating revenue uplift by rerouting high-failure payments to alternative healthy banks/methods.

In [ ]:
# Simulation scenario: For all high failure transactions (> 30% failure probability),
# evaluate rerouting to ICICI UPI or HDFC Credit Card.
test_df = X_test.copy()
test_df['actual_failed'] = y_test.values
test_df['predicted_fail_prob'] = y_prob

high_risk_routes = test_df[test_df['predicted_fail_prob'] > 0.25].copy()
print(f"Transactions flagged for smart dynamic rerouting: {len(high_risk_routes)}")

# Simulate route switch to alternate high-availability gateway
simulated_alternate = high_risk_routes.copy()
simulated_alternate['bank_code'] = 'ICICI'
simulated_alternate['payment_method'] = 'upi'

rerouted_fail_prob = failure_pipeline.predict_proba(simulated_alternate[FEATURES])[:, 1]
avg_original_success = (1 - high_risk_routes['predicted_fail_prob']).mean() * 100
avg_rerouted_success = (1 - rerouted_fail_prob).mean() * 100

uplift = avg_rerouted_success - avg_original_success

print(f"Original Channel Success Rate: {avg_original_success:.1f}%")
print(f"Rerouted Channel Success Rate: {avg_rerouted_success:.1f}%")
print(f"Net Conversion Uplift through Smart Routing: +{uplift:.1f}%")

## 5. RFM Customer Churn Modeling
Segmenting merchant customer base to identify accounts at risk of churning.

In [ ]:
customers_path = os.path.join("..", "data", "raw", "customers.csv")
cust_df = pd.read_csv(customers_path)

# Calculate aggregate transaction metrics per customer
cust_tx = df.groupby('customer_id').agg(
    total_orders=('transaction_id', 'count'),
    total_spent=('amount', lambda x: x[df.loc[x.index, 'status'] == 'captured'].sum()),
    latest_order=('created_at', 'max')
).reset_index()

merged_cust = pd.merge(cust_df, cust_tx, on='customer_id', how='left').fillna(0)

# High churn risk: churn_score > 0.70 and order count > 0
at_risk = merged_cust[merged_cust['churn_score'] > 0.70].sort_values('total_spent', ascending=False)

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=merged_cust, 
    x='total_spent', 
    y='churn_score', 
    hue=merged_cust['churn_score'] > 0.70,
    palette=['#10b981', '#ef4444'],
    alpha=0.7
)
plt.title("Customer Spend vs Churn Probability (Red = At Risk Cohort)", fontsize=13, fontweight='bold')
plt.xlabel("Total Lifetime Spend (INR)")
plt.ylabel("Churn Risk Score")
plt.show()

print(f"Identified {len(at_risk)} at-risk customers representing ₹{at_risk['total_spent'].sum():,.2f} in historical revenue.")
at_risk.head()

## 6. Exporting Production Pipeline

In [ ]:
artifact_dir = os.path.join("..", "backend", "artifacts")
os.makedirs(artifact_dir, exist_ok=True)
export_path = os.path.join(artifact_dir, "failure_model.joblib")

joblib.dump(failure_pipeline, export_path)
print(f"Production Failure Pipeline exported to: {export_path}")